In [ ]:
import torch
import pandas as pd
import numpy as np

# 加载保存的预测结果
prostab_result = torch.load('/home/xy_th/SPURS/data/checkpoints/fusn320/pred_result.pt')
spurs_results = torch.load('/home/xy_th/SPURS/data/checkpoints/spurs/pred_result.pt')
thermompnn_results = torch.load('/home/xy_th/SPURS/data/checkpoints/ThermoMPNN/pred_result.pt')


# 解包保存的结果
pred_prostab_scores, fermi_prostab_scores,_,_= prostab_result
pred_spurs_scores, fermi_spurs_scores,_,_= spurs_results
pred_thermo_scores, fermi_thermo_scores,_,_= thermompnn_results

data_prostab = {
    "Actual_DDG": fermi_prostab_scores.cpu().numpy() if isinstance(fermi_prostab_scores, torch.Tensor) else np.array(fermi_prostab_scores),
    "Predicted_DDG": pred_prostab_scores.cpu().numpy() if isinstance(pred_prostab_scores, torch.Tensor) else np.array(pred_prostab_scores)
}

data_spurs = {
    "Actual_DDG": fermi_spurs_scores.cpu().numpy() if isinstance(fermi_spurs_scores, torch.Tensor) else np.array(fermi_spurs_scores),
    "Predicted_DDG": pred_spurs_scores.cpu().numpy() if isinstance(pred_spurs_scores, torch.Tensor) else np.array(pred_spurs_scores)
}


data_thermo = {
    "Actual_DDG": fermi_thermo_scores.cpu().numpy() if isinstance(fermi_thermo_scores, torch.Tensor) else np.array(fermi_thermo_scores),
    "Predicted_DDG": pred_thermo_scores.cpu().numpy() if isinstance(pred_thermo_scores, torch.Tensor) else np.array(pred_thermo_scores)
}


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import precision_score, recall_score

# 设置全局字体大小和粗度
plt.rcParams['font.weight'] = 'bold'
plt.rcParams['axes.labelweight'] = 'bold'
plt.rcParams['axes.titleweight'] = 'bold'

# 假设你已经从模型中提取出对应的真实值与预测值
cutoffs = np.linspace(-1, 0, 5)
spurs_true = data_spurs['Actual_DDG']
spurs_pred = data_spurs['Predicted_DDG']

thermo_true = data_thermo['Actual_DDG']
thermo_pred = data_thermo['Predicted_DDG']

prostab_true = data_prostab['Actual_DDG']
prostab_pred = data_prostab['Predicted_DDG']

precision_spurs = []
recall_spurs = []
precision_thermo = []
recall_thermo = []
precision_prostab = []
recall_prostab = []

for c in cutoffs:
    true_binary = spurs_true < c
    pred_binary_spurs = spurs_pred < c
    pred_binary_thermo = thermo_pred < c
    pred_binary_prostab = prostab_pred < c
    
    precision_spurs.append(precision_score(true_binary, pred_binary_spurs))
    recall_spurs.append(recall_score(true_binary, pred_binary_spurs))

    precision_thermo.append(precision_score(true_binary, pred_binary_thermo))
    recall_thermo.append(recall_score(true_binary, pred_binary_thermo))
    
    precision_prostab.append(precision_score(true_binary, pred_binary_prostab))
    recall_prostab.append(recall_score(true_binary, pred_binary_prostab))

# 第一个图：Precision
plt.figure(figsize=(8, 6))
plt.plot(cutoffs, precision_spurs, marker='s', label='SPURS', color='blue')
plt.plot(cutoffs, precision_thermo, marker='o', label='ThermoMPNN', color='orange')
plt.plot(cutoffs, precision_prostab, marker='^', label='ProStab', color='green')
plt.ylim(0.1, 1.0)
plt.xlabel('ΔΔG Cutoff')
plt.ylabel('Precision')
plt.title('Megascale')
plt.legend()
plt.tick_params(width=1.5, length=6, colors='black')  # 加粗刻度线
plt.tight_layout()
plt.savefig('./fig/Megascale_precision.pdf')
plt.show()

# 第二个图：Recall
plt.figure(figsize=(8, 6))
plt.plot(cutoffs, recall_spurs, marker='s', label='SPURS', color='blue')
plt.plot(cutoffs, recall_thermo, marker='o', label='ThermoMPNN', color='orange')
plt.plot(cutoffs, recall_prostab, marker='^', label='ProStab', color='green')
plt.ylim(0.0, 0.7)
plt.xlabel('ΔΔG Cutoff')
plt.ylabel('Recall')
plt.title('Megascale')
plt.legend()
plt.tick_params(width=1.5, length=6, colors='black')  # 加粗刻度线
plt.tight_layout()
plt.savefig('./fig/Megascale_recall.pdf')
plt.show()